# 08 小波分解 + 双向LSTM (Wavelet-BiLSTM) 日内收益预测

## 论文参考

- **作者**: Da-tung Liang (梁大同)
- **年份**: 2023
- **标题**: *Intraday Return Forecasts and High-Frequency Trading of Stock Index Futures*

### 摘要

金融时间序列包含多种频率成分（长期趋势、中期波动、短期噪声），直接建模效果有限。
本文提出先用Daubechies小波将价格序列分解为不同频率成分（近似分量+细节分量），
然后为每个频率分量分别训练一个Bi-LSTM模型，最后将各分量预测结果加和得到最终预测。
该方法有效分离了信号中的趋势和噪声，提升了预测精度。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 离散小波变换 (DWT)

使用Daubechies-4 (db4) 小波基函数，将信号 $x(t)$ 分解为 $L$ 层：

$$x(t) = A_L(t) + \sum_{l=1}^{L} D_l(t)$$

其中 $A_L$ 为第 $L$ 层近似分量（低频趋势），$D_l$ 为第 $l$ 层细节分量（高频波动）。

### 分解过程

每层分解通过低通滤波器 $h$ 和高通滤波器 $g$ 实现：

$$A_l[n] = \sum_k h[k] \cdot A_{l-1}[2n-k] \quad \text{(近似系数)}$$
$$D_l[n] = \sum_k g[k] \cdot A_{l-1}[2n-k] \quad \text{(细节系数)}$$

本文使用3层分解 ($L=3$)，得到 $[A_3, D_3, D_2, D_1]$ 共4个分量。

### Bi-LSTM 模型

对每个分量分别建立Bi-LSTM：

$$\overrightarrow{h_t} = \text{LSTM}_{\text{forward}}(x_t, \overrightarrow{h_{t-1}})$$
$$\overleftarrow{h_t} = \text{LSTM}_{\text{backward}}(x_t, \overleftarrow{h_{t+1}})$$
$$h_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]$$

### 预测合成

$$\hat{y} = \text{sign}\left(\hat{A}_3 + \hat{D}_3 + \hat{D}_2 + \hat{D}_1\right)$$

In [ ]:
# ============================================================
# Cell 3: 动画 - 小波分解过程可视化
# ============================================================
import numpy as np
import pywt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def animate_wavelet_decomposition():
    """动画展示小波分解将信号逐级拆分为近似+细节分量"""
    np.random.seed(42)
    t = np.linspace(0, 4 * np.pi, 512)
    # 模拟金融信号: 趋势 + 中频波动 + 高频噪声
    trend = 0.5 * np.sin(0.3 * t) + 0.02 * t
    mid_freq = 0.3 * np.sin(3 * t)
    high_freq = 0.15 * np.sin(12 * t)
    noise = 0.1 * np.random.randn(len(t))
    signal = trend + mid_freq + high_freq + noise

    # 小波分解
    coeffs = pywt.wavedec(signal, 'db4', level=3)
    # 重构各分量
    components = []
    labels = ['A3 (低频趋势)', 'D3 (中低频)', 'D2 (中高频)', 'D1 (高频噪声)']
    for i in range(len(coeffs)):
        c = [np.zeros_like(c_) for c_ in coeffs]
        c[i] = coeffs[i]
        rec = pywt.waverec(c, 'db4')[:len(signal)]
        components.append(rec)

    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    frames = []

    # Frame 0: 原始信号
    frames.append(go.Frame(
        data=[go.Scatter(x=t, y=signal, mode='lines',
                         line=dict(color='#333', width=1.5), name='原始信号')],
        name='原始信号'
    ))

    # 逐步显示分解结果
    for step in range(1, len(components) + 1):
        data = [go.Scatter(x=t, y=signal, mode='lines',
                           line=dict(color='rgba(100,100,100,0.3)', width=1), name='原始信号')]
        for j in range(step):
            data.append(go.Scatter(
                x=t, y=components[j] - (j + 1) * 1.5,
                mode='lines', line=dict(color=colors[j], width=1.5),
                name=labels[j]
            ))
        frames.append(go.Frame(data=data, name=f'分解: {labels[step-1]}'))

    # 最终: 重构验证
    reconstruction = sum(components)
    final_data = [
        go.Scatter(x=t, y=signal, mode='lines',
                   line=dict(color='#333', width=1.5), name='原始信号'),
        go.Scatter(x=t, y=reconstruction, mode='lines',
                   line=dict(color='red', width=1, dash='dash'), name='重构信号'),
    ]
    for j in range(len(components)):
        final_data.append(go.Scatter(
            x=t, y=components[j] - (j + 1) * 1.5,
            mode='lines', line=dict(color=colors[j], width=1.2),
            name=labels[j]
        ))
    frames.append(go.Frame(data=final_data, name='重构验证 (A3+D3+D2+D1)'))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title=dict(text='Daubechies-4 小波分解动画 (3层分解: A3 + D3 + D2 + D1)'),
        xaxis_title='时间', yaxis_title='幅值',
        height=600, width=1000,
        plot_bgcolor='white',
        updatemenus=[dict(type='buttons', x=0.1, y=1.08, buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 1500}, 'transition': {'duration': 500}}]),
            dict(label='\u23f8 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0, x=0.05, len=0.9,
        )],
    )
    return fig

fig = animate_wavelet_decomposition()
fig.show()

In [ ]:
# ============================================================
# Cell 4: 导入与环境设置
# ============================================================
import sys, os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import pywt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from shared.data_fetcher import get_stock_daily
from shared.backtest_engine import Backtester
from shared.visualization import plot_full_report, set_chinese_font
from shared.factors import momentum, volatility, rsi, macd, bollinger_bands

set_chinese_font()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
SYMBOL = '000858'  # 五粮液
df = get_stock_daily(SYMBOL, start_date='20200601', end_date='20241231')
print(f'数据形状: {df.shape}')
print(f'日期范围: {df.index[0]} ~ {df.index[-1]}')
df.tail(3)

In [ ]:
# ============================================================
# Cell 6: 特征工程 - 小波分解 + 序列构建
# ============================================================
LOOKBACK = 30
WAVELET = 'db4'
LEVEL = 3

def wavelet_decompose_series(series, wavelet='db4', level=3):
    """对价格序列进行小波分解，返回各分量"""
    coeffs = pywt.wavedec(series, wavelet, level=level)
    components = []
    for i in range(len(coeffs)):
        c = [np.zeros_like(c_) for c_ in coeffs]
        c[i] = coeffs[i]
        rec = pywt.waverec(c, wavelet)
        # 对齐长度
        rec = rec[:len(series)]
        components.append(rec)
    return components  # [A_L, D_L, D_{L-1}, ..., D_1]


def build_wavelet_dataset(df, lookback=30, wavelet='db4', level=3):
    """构建小波分解的多通道序列数据集"""
    close = df['close'].values
    returns = df['close'].pct_change().fillna(0).values

    # 对整段收益率序列做小波分解
    components = wavelet_decompose_series(returns, wavelet, level)
    n_components = len(components)  # level+1 = 4

    # 附加特征: 成交量变化率
    vol_change = df['volume'].pct_change().fillna(0).values

    # 构建滑窗序列
    X_components = [[] for _ in range(n_components)]  # 每个分量一个列表
    y_list = []
    dates = []

    for i in range(lookback, len(close) - 1):
        for ci in range(n_components):
            # 每个分量: (lookback, 2) -> 分量值 + 成交量
            comp_seq = np.column_stack([
                components[ci][i-lookback:i],
                vol_change[i-lookback:i]
            ])
            X_components[ci].append(comp_seq)

        # 标签: 次日涨跌
        y_list.append(1 if close[i+1] > close[i] else 0)
        dates.append(df.index[i])

    X_comp_arrays = [np.array(xc, dtype=np.float32) for xc in X_components]
    y_array = np.array(y_list, dtype=np.float32)
    return X_comp_arrays, y_array, dates


X_components, y, dates = build_wavelet_dataset(df, LOOKBACK, WAVELET, LEVEL)
print(f'分量数: {len(X_components)}')
for i, xc in enumerate(X_components):
    lbl = 'A3' if i == 0 else f'D{LEVEL - i + 1}'
    print(f'  {lbl}: shape={xc.shape}')
print(f'标签: {len(y)}, 涨={int(y.sum())}, 跌={int(len(y) - y.sum())}')

In [ ]:
# ============================================================
# Cell 7: PyTorch Wavelet-BiLSTM 模型与训练
# ============================================================

class BiLSTMBranch(nn.Module):
    """单个频率分量的Bi-LSTM分支"""
    def __init__(self, input_dim, hidden_dim=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True, dropout=0.1)
        self.fc = nn.Linear(hidden_dim * 2, 16)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # 取最后时刻
        return torch.relu(self.fc(out))


class WaveletBiLSTM(nn.Module):
    """小波分解 + 分支Bi-LSTM + 合并预测"""
    def __init__(self, input_dim, n_components=4, hidden_dim=32):
        super().__init__()
        self.branches = nn.ModuleList([
            BiLSTMBranch(input_dim, hidden_dim) for _ in range(n_components)
        ])
        self.merge = nn.Sequential(
            nn.Linear(16 * n_components, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, components):
        """components: list of tensors, each (batch, seq_len, input_dim)"""
        branch_outs = [branch(comp) for branch, comp in zip(self.branches, components)]
        merged = torch.cat(branch_outs, dim=-1)
        return self.merge(merged).squeeze(-1)


# --- 数据划分 ---
TRAIN_END = pd.Timestamp('2023-12-31')
dates_ts = pd.DatetimeIndex(dates)
train_mask = dates_ts <= TRAIN_END
test_mask = dates_ts > TRAIN_END

# 标准化每个分量
scalers = []
X_train_comp, X_test_comp = [], []
for xc in X_components:
    sc = StandardScaler()
    n, seq, feat = xc.shape
    train_flat = xc[train_mask].reshape(-1, feat)
    sc.fit(train_flat)
    scalers.append(sc)

    train_scaled = sc.transform(xc[train_mask].reshape(-1, feat)).reshape(-1, seq, feat)
    test_scaled = sc.transform(xc[test_mask].reshape(-1, feat)).reshape(-1, seq, feat)
    X_train_comp.append(torch.FloatTensor(train_scaled))
    X_test_comp.append(torch.FloatTensor(test_scaled))

y_train = torch.FloatTensor(y[train_mask])
y_test = y[test_mask]
test_dates = dates_ts[test_mask]

print(f'训练: {len(y_train)} 样本, 测试: {len(y_test)} 样本')

# --- DataLoader ---
train_dataset = TensorDataset(*X_train_comp, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 训练 ---
n_comp = len(X_components)
input_dim = X_components[0].shape[2]  # 2
model = WaveletBiLSTM(input_dim=input_dim, n_components=n_comp, hidden_dim=32).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_samples = 0
    for batch in train_loader:
        comps = [b.to(device) for b in batch[:n_comp]]
        labels = batch[n_comp].to(device)
        optimizer.zero_grad()
        pred = model(comps)
        loss = criterion(pred, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(labels)
        n_samples += len(labels)
    scheduler.step()

    avg_loss = epoch_loss / max(n_samples, 1)
    train_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}')

# --- 预测 ---
model.eval()
with torch.no_grad():
    test_comps = [xc.to(device) for xc in X_test_comp]
    test_probs = model(test_comps).cpu().numpy()

test_preds = (test_probs > 0.5).astype(int)
accuracy = (test_preds == y_test).mean()
print(f'\n测试集准确率: {accuracy:.4f}')

In [ ]:
# ============================================================
# Cell 8: 回测
# ============================================================
signals = pd.Series(test_preds, index=test_dates, name='signal')
test_prices = df.loc[test_dates, 'close']

bt = Backtester(initial_capital=1_000_000, t_plus_1=True)
result = bt.run(test_prices, signals)

print('回测绩效指标:')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 小波分解展示
close_ret = df['close'].pct_change().dropna().values[-200:]
comps = wavelet_decompose_series(close_ret, WAVELET, LEVEL)
comp_labels = ['A3 (低频趋势)', 'D3 (中低频)', 'D2 (中高频)', 'D1 (高频噪声)']

fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
axes[0].plot(close_ret, color='#333', linewidth=1)
axes[0].set_title('原始收益率序列', fontsize=11)
axes[0].grid(True, alpha=0.3)

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
for i, (comp, label) in enumerate(zip(comps, comp_labels)):
    axes[i+1].plot(comp, color=colors[i], linewidth=1)
    axes[i+1].set_title(label, fontsize=11)
    axes[i+1].grid(True, alpha=0.3)

fig.suptitle('Daubechies-4 小波3层分解', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 2. 训练损失
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, color='#2196F3', linewidth=1.5)
ax.set_title('Wavelet-BiLSTM 训练损失', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. 完整回测报告
plot_full_report(result, strategy_name='Wavelet-BiLSTM (Liang 2023)')

## 结果分析

### 模型架构

- **小波分解**: Daubechies-4, 3层 -> A3(趋势) + D3(中低频) + D2(中高频) + D1(噪声)
- **4个Bi-LSTM分支**: 每个分支独立处理一个频率分量
- **合并层**: 拼接4个分支输出 -> FC(32) -> Sigmoid
- **输入窗口**: 30个交易日

### 关键优势

1. **频率分离**: 小波分解将趋势和噪声分开，让模型专注于有意义的信号
2. **Bi-LSTM**: 双向结构在序列建模中能捕获更丰富的上下文信息
3. **分支独立**: 各频率分量由独立模型处理，避免相互干扰

### 局限性

- 小波分解的边界效应可能影响最新数据点的预测质量
- db4小波的选择和分解层数(L=3)需要根据数据特性调参
- 多个Bi-LSTM分支增加了模型复杂度和训练时间